In [1]:
!pip install chromadb langchain-google-genai langchain-community google-generativeai langchain

  Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   --------------------------- ------------ 2.4/3.5 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 14.7 MB/s eta 0:00:00
Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl (183 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.64 requires beautifulsoup4>=4.11.1, which is not installed.
yfinance 0.2.64 requires frozendict>=2.3.4, which is not installed.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\deppa\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()

# API Key set करें
# os.environ["GOOGLE_API_KEY"] = "your-gemini-api-key-here"

d:\Agenti Ai\Langchain campusX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
# Sample documents — आप यहाँ अपना data डाल सकते हैं
raw_texts = [
    """
    Virat Kohli is one of the greatest batsmen in cricket history.
    He plays for Royal Challengers Bangalore in IPL.
    He is known for his aggressive batting style and fitness.
    He has scored more than 7000 runs in IPL.
    """,
    """
    Rohit Sharma is the captain of Mumbai Indians in IPL.
    He is known for his elegant batting and leadership skills.
    He has won IPL trophy 5 times with Mumbai Indians.
    He is also the captain of Indian national cricket team.
    """,
    """
    MS Dhoni is the captain of Chennai Super Kings in IPL.
    He is known for his calm and strategic leadership.
    He has won IPL trophy 5 times with Chennai Super Kings.
    He is famous for his finishing skills and wicket keeping.
    """,
    """
    Jasprit Bumrah is the best fast bowler in Indian cricket team.
    He plays for Mumbai Indians in IPL.
    He is known for his unique bowling action and yorkers.
    He is considered one of the best death bowlers in the world.
    """,
    """
    Ravindra Jadeja is an all-rounder who plays for Chennai Super Kings.
    He can both bat and bowl effectively.
    He is a left-arm spinner and a hard-hitting lower-order batsman.
    He is also an excellent fielder in the Indian cricket team.
    """
]

# Text splitter 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,       
    chunk_overlap=50      
)

# Documents  split
documents = []
for i, text in enumerate(raw_texts):
    docs = text_splitter.create_documents(
        texts=[text],
        metadatas=[{"source": f"player_{i+1}"}]
    )
    documents.extend(docs)

print(f"✅ Total chunks created: {len(documents)}")

print(documents[0].page_content)
print("\n📄 Sample Chunk:")
print(documents)
print("🏷️ Metadata:", documents[0].metadata)

✅ Total chunks created: 10
Virat Kohli is one of the greatest batsmen in cricket history.
    He plays for Royal Challengers Bangalore in IPL.
    He is known for his aggressive batting style and fitness.

📄 Sample Chunk:
[Document(metadata={'source': 'player_1'}, page_content='Virat Kohli is one of the greatest batsmen in cricket history.\n    He plays for Royal Challengers Bangalore in IPL.\n    He is known for his aggressive batting style and fitness.'), Document(metadata={'source': 'player_1'}, page_content='He has scored more than 7000 runs in IPL.'), Document(metadata={'source': 'player_2'}, page_content='Rohit Sharma is the captain of Mumbai Indians in IPL.\n    He is known for his elegant batting and leadership skills.\n    He has won IPL trophy 5 times with Mumbai Indians.'), Document(metadata={'source': 'player_2'}, page_content='He is also the captain of Indian national cricket team.'), Document(metadata={'source': 'player_3'}, page_content='MS Dhoni is the captain of Chenna

In [5]:
# Gemini Embedding Model
embeddings = GoogleGenerativeAIEmbeddings(
     model="gemini-embedding-001"
)

# Test करें — एक text का embedding देखें
sample_vector = embeddings.embed_query("Who is Virat Kohli?")
print(f"✅ Embedding dimension: {len(sample_vector)}")
print(f"📊 Sample values: {sample_vector[:5]}")  # पहले 5 values

✅ Embedding dimension: 3072
📊 Sample values: [-0.0035524718, -0.009195617, 0.02712564, -0.07105515, -0.019352611]


In [ ]:
# Vector Store बनाएं — documents + embeddings
import chromadb
import chromadb.config

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./cricket_chroma_db",   # disk पर save होगा
    collection_name="cricket_players"
)

print("✅ Vector Store created successfully!")
print(f"📦 Total vectors stored: {vectorstore._collection.count()}")

ImportError: Could not import chromadb python package. Please install it with `pip install chromadb`.

In [ ]:
# Retriever — vector store से relevant docs निकालेगा
retriever = vectorstore.as_retriever(
    search_type="similarity",   # similarity search
    search_kwargs={"k": 3}      # top 3 relevant chunks
)

# Test करें
test_query = "Who is the best bowler?"
relevant_docs = retriever.invoke(test_query)

print(f"\n🔍 Query: {test_query}")
print(f"📄 Retrieved {len(relevant_docs)} documents:\n")
for i, doc in enumerate(relevant_docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print(f"Source: {doc.metadata}")
    print()

In [ ]:
# Gemini Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",   # या "gemini-pro"
    temperature=0.3
)

# RAG Prompt
rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful cricket expert assistant.
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:
""")

In [ ]:
# Context format करने का function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Complete RAG Chain
rag_chain = (
    {
        "context": retriever | format_docs,  # retriever → context
        "question": RunnablePassthrough()    # question as-is
    }
    | rag_prompt     # prompt में डालो
    | llm            # LLM को भेजो
    | StrOutputParser()  # output parse करो
)

print("✅ RAG Chain ready!")

In [ ]:
# Question 1
question1 = "Who is the best bowler among these players?"
answer1 = rag_chain.invoke(question1)
print(f"❓ Q: {question1}")
print(f"✅ A: {answer1}\n")

# Question 2
question2 = "Which players play for Chennai Super Kings?"
answer2 = rag_chain.invoke(question2)
print(f"❓ Q: {question2}")
print(f"✅ A: {answer2}\n")

# Question 3
question3 = "Who has won the most IPL trophies?"
answer3 = rag_chain.invoke(question3)
print(f"❓ Q: {question3}")
print(f"✅ A: {answer3}\n")

In [ ]:
# अगर आप program बंद करके दोबारा load करना चाहते हैं
vectorstore_loaded = Chroma(
    persist_directory="./cricket_chroma_db",
    embedding_function=embeddings,
    collection_name="cricket_players"
)

print(f"✅ Loaded vectors: {vectorstore_loaded._collection.count()}")

In [ ]:
# ➕ नए documents add करें
new_doc = Document(
    page_content="Hardik Pandya is an all-rounder who plays for Mumbai Indians.",
    metadata={"source": "player_6"}
)
vectorstore.add_documents([new_doc])
print("✅ Document added!")

# 🔎 Similarity Search with Score
results_with_score = vectorstore.similarity_search_with_score(
    query="Who is an all-rounder?",
    k=2
)
for doc, score in results_with_score:
    print(f"📄 Content: {doc.page_content[:100]}")
    print(f"📊 Distance Score: {score}")  # कम score = ज्यादा similar
    print("---")

# 🏷️ Metadata Filter के साथ Search
filtered_results = vectorstore.similarity_search(
    query="",
    k=5,
    filter={"source": "player_3"}   # सिर्फ player_3 के docs
)
for doc in filtered_results:
    print(doc.page_content)

In [ ]:
!pip install chromadb


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\deppa\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [ ]:
!pip install chromadb langchain langchain-community


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\deppa\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import os
from dotenv import load_dotenv

load_dotenv()
# Set API Key
# os.environ["GOOGLE_API_KEY"] = "your-gemini-api-key-here"

# Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# Create sample docs
docs = [
    Document(page_content="Virat Kohli is a legendary batsman.", metadata={"team": "Royal Challengers Bangalore"}),
    Document(page_content="Rohit Sharma is the captain of Mumbai Indians.", metadata={"team": "Mumbai Indians"}),
    Document(page_content="MS Dhoni is a calm and strategic leader.", metadata={"team": "Chennai Super Kings"}),
]

# Create Chroma vector store
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./my_chroma_db",
    collection_name="cricket_players"
)

# Search
query = "Who is a good captain?"
results = vectorstore.similarity_search(query, k=2)

for doc in results:
    print("👉", doc.page_content)
    print("🏷️ Metadata:", doc.metadata)
    print("---")

ImportError: Could not import chromadb python package. Please install it with `pip install chromadb`.

In [ ]:
import sys
print(sys.executable)

d:\Agenti Ai\Langchain campusX\.venv\Scripts\python.exe


In [ ]:
import sys
!{sys.executable} -m pip install chromadb

'd:\Agenti' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import chromadb
print("chromadb installed successfully ✅")

ModuleNotFoundError: No module named 'chromadb'